# Notebook 02 — Phase 2 Social Learning Demo

This notebook walks through the Phase 2 pipeline interactively:

1. Load Phase 1 transcripts from disk
2. Score a sample of negotiations with the **Observer agent**
3. Select the top-performing runs per role
4. Extract a **reusable tactic** (description + few-shot example) for each role
5. Inspect the resulting Phase 2 system prompts (prompt-based behavioral cloning)
6. Run one Phase 2 negotiation with the learned tactic injected

The full Phase 2 batch is run via `experiments/run_phase2.py`. This notebook is for explanation.


> **Prerequisite:** a completed Phase 1 run. Set `PHASE1_RUN_ID` below to its run_id.

## Setup

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
import json, uuid
from pathlib import Path

from src.utils.deepseek_client import DeepSeekClient
from src.agents.observer_agent import ObserverAgent
from src.agents.negotiating_agent import NegotiatingAgent
from src.simulation.session import NegotiationSession
from src.simulation.dialogue_loop import load_config
from src.social_learning.tactic_extractor import TacticExtractor
from src.social_learning.prompt_updater import PromptUpdater
from src.logging.transcript_logger import TranscriptLogger

# >>> EDIT THIS <<<
PHASE1_RUN_ID = "20260518_213850"   # change to your real Phase 1 run id

PHASE1_DIR = Path("data/raw") / PHASE1_RUN_ID
assert PHASE1_DIR.exists(), f"Phase 1 run directory not found: {PHASE1_DIR}"
print(f"Loaded Phase 1 run: {PHASE1_RUN_ID}")
print(f"Number of sessions: {len(list(PHASE1_DIR.glob('*.json')))}")

## 1. Look at one Phase 1 session

The Observer takes the rendered transcript (visible turns only — never the CoT) plus the persona metadata.

In [ ]:
sample_path = sorted(PHASE1_DIR.glob("*.json"))[0]
sample = TranscriptLogger.load_session(str(sample_path))
print(TranscriptLogger.format_transcript(sample))

## 2. Score that session with the Observer

`ObserverAgent.score_negotiation()` calls DeepSeek-R1 with a low temperature and asks for a structured JSON response: outcome, seller/buyer scores (1–5) with rationales, and notable tactics.

In [ ]:
client = DeepSeekClient()
observer = ObserverAgent(client=client)

scenario = load_config("config/scenarios.yaml")["software_sale"]
score_json = observer.score_negotiation(
    transcript=TranscriptLogger.format_transcript(sample),
    scenario_description=scenario["description"].strip(),
    seller_persona=sample["seller_persona"],
    buyer_persona=sample["buyer_persona"],
)
try:
    parsed = json.loads(score_json)
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("(model did not return valid JSON — raw response below)")
    print(score_json)

## 3. Score ALL Phase 1 sessions + extract tactics

`TacticExtractor.extract_tactics()` is the full Phase-2 step-1 pipeline:

1. Score every Phase 1 session
2. Pick the top-k per role (default `top_k=2`)
3. Ask the Observer to extract a reusable tactic description + example for each role

This is expensive (one Observer call per session + 2 extraction calls). If you have already produced `data/processed/learned_tactics.json` via `python -m experiments.run_phase2`, you can skip this and just load it (see the next cell).

In [ ]:
updater = PromptUpdater()

if updater.tactics_exist():
    print("Found cached tactics — loading from disk.")
    tactics = updater.load_tactics()
else:
    print("No cached tactics — running the full extraction pipeline (this may take a few minutes)...")
    extractor = TacticExtractor(observer=observer)
    tactics = extractor.extract_tactics(run_dir=str(PHASE1_DIR), top_k=2)
    updater.save_tactics(tactics)

for role, t in tactics.items():
    print(f"\n=== {role.upper()} TACTIC ===")
    print("DESCRIPTION:")
    print(t["description"])
    print("\nEXAMPLE:")
    print(t["example"][:800] + ("\n...[truncated]" if len(t["example"]) > 800 else ""))

## 4. Prompt-based behavioral cloning — diff the prompts

The learned tactic is injected into the agent's system prompt via the `TACTIC_SECTION_TEMPLATE` in `negotiating_agent.py`. We build a Phase-1 agent (no tactic) and a Phase-2 agent (with tactic) side by side and look at the difference.

In [ ]:
personas_config = load_config("config/personas.yaml")
cfg = personas_config["configurations"]["B"]
seller_persona = personas_config[cfg["seller"]]
buyer_persona  = personas_config[cfg["buyer"]]

seller_p1 = NegotiatingAgent(role="seller", scenario=scenario, persona=seller_persona, client=client)
seller_p2 = NegotiatingAgent(role="seller", scenario=scenario, persona=seller_persona, client=client,
                             learned_tactic=tactics["seller"])

import difflib
from IPython.display import HTML, display

diff = difflib.HtmlDiff(wrapcolumn=80).make_table(
    seller_p1.system_prompt.splitlines(),
    seller_p2.system_prompt.splitlines(),
    fromdesc="Phase 1 prompt",
    todesc="Phase 2 prompt (with learned tactic)",
)
display(HTML(diff))

## 5. Run one Phase 2 negotiation

Both agents now carry the role-specific learned tactic in their system prompt.

In [ ]:
buyer_p2 = NegotiatingAgent(role="buyer", scenario=scenario, persona=buyer_persona, client=client,
                            learned_tactic=tactics["buyer"])

session = NegotiationSession(
    session_id=str(uuid.uuid4()),
    config_name="B",
    seller=seller_p2,
    buyer=buyer_p2,
    max_turns=scenario.get("max_turns", 15),
    phase=2,
)
p2_outcome = session.run()
print(f"Outcome: {p2_outcome.outcome}")
print(f"Turns:   {p2_outcome.turns}")
print(f"Price:   {p2_outcome.final_price}")
print(f"Bug disclosed (seller):  {p2_outcome.bug_disclosed}")
print(f"Bug discovered (buyer):  {p2_outcome.bug_discovered}")

In [ ]:
from IPython.display import Markdown, display

lines = []
for turn in p2_outcome.transcript:
    icon = "\U0001f4b0" if turn["role"] == "seller" else "\U0001f6d2"
    lines.append(f"**{icon} Turn {turn['turn']} — {turn['role'].upper()}**")
    lines.append(f"> {turn['content']}")
    lines.append("")
display(Markdown("\n".join(lines)))

## 6. Running the full Phase 2 batch

```bash
# Full pipeline: score all Phase 1, extract tactics, run 80 negotiations
python -m experiments.run_phase2 --phase1-run-id <phase1_run_id> --runs 20 --workers 10

# Or skip tactic extraction if learned_tactics.json already exists
python -m experiments.run_phase2 --phase1-run-id <phase1_run_id> --skip-extraction --workers 10
```

Phase-1-vs-Phase-2 comparison is the topic of `03_analysis_visualization.ipynb`.